In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.options.display.max_columns = 500

### Загрузим датасет с машинами. Цель - верно восстанавливать для каждой из них цену продажи!

In [2]:
data = pd.read_csv('autos.csv')

data.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


In [3]:
### Колонка с тергетом - "selling price"

X = data.drop("selling_price", axis=1)
y = data["selling_price"]

### Будем замерять MSLE!
### Поэтому прологарифмируем таргет
### А после оптимизируем MSE

y = y.apply(np.log1p)

In [4]:
### Разделим выборку на трейн и тест!

from sklearn.model_selection import train_test_split 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

__Задание__ 

Реализуйте свой MeanTargetEncoder с добавленем некоторого шума!

Однажды в лекционном материале, обсуждая счетчики, мы говорили с вами о том, что из-за них модели могут переобучаться. Один из способов бороться с этим - валидировать расчеты среднего таргета (стратегия отложенной выборки / расчеты на кросс-валидации). Но есть еще проще!

Можно просто к значению счетчика добавить случайный шум (зашумить данные)!

Напомним, что рассчитываться новые признаки должны по такой формуле:

$$
g_j = \frac{\sum_{i=1}^{l} [f_j(x) = f_j(x_i)]}{l} + C * \epsilon
$$



Пусть шум будет случайной величиной из нормального стандартного распределения, то есть $\epsilon \sim N(0, 1) $, а $ C = 0.006$.

Создавая свой класс-трансформер, наследуйтесь от классов `BaseEstimator, TransformerMixin` из `sklearn.base`. Трансформер не должен модифицировать передаваемую ему выборку inplace, а все необходимые статистики нужно считать только по обучающей выборке в методе `fit`. Ваш трансформер должен принимать при инициализации список из категориальных признаков и список из числовых признаков. 

Если для какого-то признака в тестовой выборке отсутствует значение, трансформер должен поставить там 0.

На выходе должен получиться датасет того же размера с измененными категориальными признаками.

Класс MeanTargetEncoderNoise должен иметь следующую сигнатуру:



In [5]:
from sklearn.base import BaseEstimator, TransformerMixin

class MeanTargetEncoderNoise(BaseEstimator, TransformerMixin):
    
    def __init__(self, categorical, numeric):
        
        ### Your code is here
    
    def fit(self, X, y):

        ### Your code is here

        return self
        
    def transform(self, df):
        
        ### Your code is here
        
        return temp

IndentationError: expected an indented block after function definition on line 5 (1446550018.py, line 9)

Разделите колонки на вещественные и категориальные. Приведите все категориальные колонки к типу `object`.

Далее применим наш кодировщик к `X_train, X_test`, так же как например мы применяем `StandardScaler`, чтобы проверить работоспособность нашего класса. Установите зерно датчика случайный чисел `np.random.seed(1)`.

После того, как вы изменили обучающую и тестовую выборки, сохраните первые 10 строк полученного промежуточного датафрейма обучающей выборки (`X_train`) в файл в формате csv с сепаратором `;`. Не забудьте индекс. Отправьте полученный файл в форму ниже.

Список колонок которые должны быть в файле для сдачи:
```py
cols = [
    "km_driven",
    "name",
    "year",
    "fuel",
    "seller_type",
    "transmission",
    "owner"
]
```

### Ваше решение


Разделение колонок на категориальные и числовые.

In [6]:
object_cols = ['name', 'year', 'fuel', 'seller_type', 'transmission', 'owner']
num_cols = ['km_driven']

X.head()

,name,year,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,141000,Diesel,Individual,Manual,Second Owner


Реализация класса MeanTargetEncoderNoise.

In [14]:
from sklearn.base import BaseEstimator, TransformerMixin

class MeanTargetEncoderNoise(BaseEstimator, TransformerMixin):
    
    def __init__(self, categorical, numeric, target_col='selling_price'):
        self.categorical = categorical
        self.numeric = numeric
        self.target_col = target_col
        
    def fit(self, X, y):
        
        X_fit = X.copy()
        y_fit = y.copy()
        
        # Определили числовые колонки
        # self.numeric_cols = [col for col in X_fit.columns if col not in self.categorical]
        
        # Объединяем с таргетом
        X_with_target = pd.concat((X_fit, y_fit), axis=1)
        
        # Создаем словарь, каждый элемент которого - колонка (ключ) и серия с зашумленным значением таргета (значение)
        self.mte_dict = {col: X_with_target.groupby(col)[self.target_col].mean() + 0.006 * np.random.normal()
                         for col in self.categorical}
        
        # Список всех колонок с mte
        self.mte_cols = self.mte_dict.keys()
        
        return self
        
    def transform(self, df):
        
        X_ = df.copy()
        
        for col in self.categorical:
            X_[col] = X_[col].map(self.mte_dict[col])
            X_[col] = X_[col].fillna(0)
        
        return X_

Проверка работы трансформера.

In [15]:
np.random.seed(1)
transformer = MeanTargetEncoderNoise(categorical=object_cols, numeric=num_cols)

transformer.fit(X_train, y_train)

train: pd.DataFrame = transformer.transform(X_train)
test = transformer.transform(X_test)


train.head(10).to_csv("175.csv", index=False, sep=";")
train.head()

,name,year,km_driven,fuel,seller_type,transmission,owner
3294,13.478865,13.435892,50000,13.085954,12.611658,13.776443,12.959126
2290,12.125200,11.906484,70000,12.450179,12.611658,13.776443,12.959126
874,12.311508,13.333058,50000,12.450179,12.611658,12.643381,12.959126
1907,12.493842,13.053677,92198,12.450179,13.140882,12.643381,12.446379
3244,12.401697,12.857085,3240,12.450179,12.611658,12.643381,12.446379


Обучите несколько деревьев, перебирая максимальную глубину алгоритма из списка `max_depth_list`, а остальные параметры оставьте дефолтными. Выведите лучшее значение гиперпараметра. Постройте график зависимости MSLE на тестовой выборке от значения гиперпараметра. Воспользуйтесь `Pipeline` без `GridSearch`. Проделайте то же самое с `min_samples_split`, `min_impurity_decrease`, `max_leaf_nodes`. (по 2б на каждый параметр)

In [16]:
max_depth_list = [3, 5, 8, 12]
min_samples_split_list = [10, 50, 100, 500]
min_impurity_decrease_list = [0, 0.1, 0.15, 0.2]
max_leaf_nodes_list = [100, 200, 500]

In [17]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline

np.random.seed(1)

for max_depth in max_depth_list:
    pipe = Pipeline([("custom_transformer",
                   MeanTargetEncoderNoise(categorical=object_cols, numeric=num_cols)),
                  
                  
                 ("decision_tree", 
                  DecisionTreeRegressor(max_depth=max_depth))])

    pipe.fit(X_train, y_train)

    mse_test = mse(y_test, pipe.predict(X_test))
    print(max_depth, mse_test)

3 0.7968292144891994
5 1.4418633938730097
8 1.9886790473777403
12 1.9831678836075364


In [18]:
for min_samples_split in min_samples_split_list:
    pipe = Pipeline([
        ("custom_transformer", MeanTargetEncoderNoise(categorical=object_cols, numeric=num_cols)), 
        ("decision_tree", DecisionTreeRegressor(min_samples_split=min_samples_split))
    ])

    pipe.fit(X_train, y_train)

    mse_test = mse(y_test, pipe.predict(X_test))
    print(min_samples_split, mse_test)

10 1.4309046191277333
50 1.4386960572382825
100 0.9520426718791601
500 0.8076747956646517


In [22]:
for min_impurity_decrease in min_impurity_decrease_list:
    pipe = Pipeline([
        ("custom_transformer", MeanTargetEncoderNoise(categorical=object_cols, numeric=num_cols)), 
        ("decision_tree", DecisionTreeRegressor(min_impurity_decrease=min_impurity_decrease))
    ])

    pipe.fit(X_train, y_train)

    mse_test = mse(y_test, pipe.predict(X_test))
    print(min_impurity_decrease, mse_test)

0 1.9825140797725203
0.1 0.520487141303659
0.15 0.5204871413036729
0.2 0.5204871413036544


In [20]:
for max_leaf_nodes in max_leaf_nodes_list:
    pipe = Pipeline([
        ("custom_transformer", MeanTargetEncoderNoise(categorical=object_cols, numeric=num_cols)), 
        ("decision_tree", DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes))
    ])

    pipe.fit(X_train, y_train)

    mse_test = mse(y_test, pipe.predict(X_test))
    print(max_leaf_nodes, mse_test)

100 1.9883090342277725
200 1.9839205320539333
500 1.978395430303911


Подберите лучшую комбинацию параметров, используя `GridSearchCV` и набор массивов значений параметров из предыдущего задания. Для лучшей комбинации посчитайте MSLE на тестовой выборке. Получились ли лучшие параметры такими же, как если бы вы подбирали их по-отдельности при остальных гиперпараметрах по умолчанию (предыдущее задание)? (2б)

In [26]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "decision_tree__max_depth": [3, 5, 8, 12],
    "decision_tree__min_samples_split": [10, 50, 100, 500],
    "decision_tree__min_impurity_decrease": [0, 0.1, 0.15, 0.2],
    "decision_tree__max_leaf_nodes": [100, 200, 500]
}
np.random.seed(1)

pipe = Pipeline([
    ("custom_transformer", MeanTargetEncoderNoise(categorical=object_cols, numeric=num_cols)), 
    ("decision_tree", DecisionTreeRegressor())
])

search = GridSearchCV(estimator=pipe, param_grid=param_grid)

search.fit(X_train, y_train)

GridSearchCV(estimator=Pipeline(steps=[('custom_transformer',
                                        MeanTargetEncoderNoise(categorical=['name',
                                                                            'year',
                                                                            'fuel',
                                                                            'seller_type',
                                                                            'transmission',
                                                                            'owner'],
                                                               numeric=['km_driven'])),
                                       ('decision_tree',
                                        DecisionTreeRegressor())]),
             param_grid={'decision_tree__max_depth': [3, 5, 8, 12],
                         'decision_tree__max_leaf_nodes': [100, 200, 500],
                         'decision_tree__min_impurity_decrease': [0, 0.1, 0.15,
                                                                  0.2],
                         'decision_tree__min_samples_split': [10, 50, 100,
                                                              500]})

In [27]:
search.best_params_

{'decision_tree__max_depth': 8,
 'decision_tree__max_leaf_nodes': 500,
 'decision_tree__min_impurity_decrease': 0.1,
 'decision_tree__min_samples_split': 50}

In [29]:
mse(y_test, search.predict(X_test))

0.520487141303668